# Major Homework #1: Clustering

In this homework we will get to know clustering more closely.

Clustering is a machine learning method that groups data points in feature space. In theory, points in the same group should have similar properties, while points in different groups should have very different properties.

Clustering is an unsupervised learning method and a common statistical data-analysis technique used in many fields — for example, for building user profiles and for anomaly detection. In data analysis, clustering is often used to extract valuable insights by observing which groups the points fall into.

## Implementing K-means (3 points)

In this task you need to implement the K-means algorithm yourself. K-means takes vectors as input and assigns each of them a cluster number. Let's recall how the algorithm works.

For a given dataset we choose k, the number of separate groups the points belong to. These k centroids are first initialized randomly, and then their positions are optimized iteratively as follows:

1. Compute the distance from every point to every centroid.
2. Assign each point to its nearest centroid.
3. Move each centroid to the mean of the points assigned to it. If the centroids did not move, the algorithm stops; otherwise, repeat.

### Data

To evaluate our algorithm, we first generate a dataset: groups of points in two-dimensional space.
The sklearn.datasets function [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html) creates groups of two-dimensional normal distributions and assigns each point a label for the group it belongs to.

In [ ]:
import seaborn as sns
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
centers = 5 # количество групп
X_train, true_labels = make_blobs(n_samples=100, centers=centers, random_state=42)
X_train = StandardScaler().fit_transform(X_train)
sns.scatterplot(x=[X[0] for X in X_train],
                y=[X[1] for X in X_train],
                hue=true_labels,
                palette="deep",
                legend=None
                )
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### Now let's write the code

#### Helper functions

**Task** This algorithm repeatedly needs the distances between a point and a set of points. Define a function that computes Euclidean distances.

In [ ]:
import numpy as np

def euclidean(point, data):
    """
    Euclidean distance between point & data.
    Point has dimensions (m,), data has dimensions (n,m), and output will be of size (n,).
    """
    # Все вычисления должны быть выполенны с помощью numpy без использования циклов
    result = ### YOUR CODE HERE ###
    return result

#### The K-means algorithm

**Task** Complete the code

In [ ]:
from numpy.random import uniform

class KMeans:
    def __init__(self, n_clusters=8, max_iter=300):
        self.n_clusters = n_clusters # Количество кластеров
        self.max_iter = max_iter # Максимальное количество итераций

    def fit(self, X_train):
        """
        X_train: numpy array of shape (n,m)
        """
        # Случайно выбираем начальные точки центроида, равномерно распределенные по области набора данных

        # Найдите минимальное и максимальное значение для каждого измерения в X_train (координаты по широте и долготе) и сохраните их в min_ и max_
        min_, max_ = ### YOUR CODE HERE ###
        self.centroids = [uniform(min_, max_) for _ in range(self.n_clusters)]

        # Итеративно повторяем до сходимости или до максимального количества итераций
        iteration = 0
        prev_centroids = None
        while np.not_equal(self.centroids, prev_centroids).any() and iteration < self.max_iter:

            # Сортируем точки по ближайшему центроиду
            sorted_points = [[] for _ in range(self.n_clusters)]
            for x in X_train:
                dists = euclidean(x, self.centroids)
                #Найдите индекс ближайшего центроида см. функцию np.argmin
                centroid_idx = ### YOUR CODE HERE ###
                sorted_points[centroid_idx].append(x)

            # Сохраните текущие центроиды в prev_centroids
            prev_centroids = ### YOUR CODE HERE ###
            # Обновите центроиды средним значением точек в кластере
            self.centroids = ### YOUR CODE HERE ###

            for i, centroid in enumerate(self.centroids):
                if np.isnan(centroid).any():  # Если центроид не имеет ни одной точки, то выбираем предыдущее значение
                    self.centroids[i] = prev_centroids[i]
            iteration += 1

    def evaluate(self, X):
        """
        Function to evaluate the model on the dataset X.
        X: numpy array of shape (n,m)
        """
        centroids = []
        centroid_idxs = []
        for x in X:
            dists = euclidean(x, self.centroids)
            centroid_idx = np.argmin(dists)
            centroids.append(self.centroids[centroid_idx])
            centroid_idxs.append(centroid_idx)
        return centroids, centroid_idxs

### Evaluating K-means
Let's run our algorithm several times on the generated data

* Different colours show the different generated groups
* Different shapes show the different clusters
* The thin blue cross is the centroid of each cluster

In [ ]:
import warnings
warnings.filterwarnings('ignore')


for i in range(3):
    kmeans = KMeans(n_clusters=centers) # Передадим количество кластеров, которое мы задали в данных
    kmeans.fit(X_train)
    # View results
    class_centers, classification = kmeans.evaluate(X_train)
    sns.scatterplot(x=[X[0] for X in X_train],
                    y=[X[1] for X in X_train],
                    hue=true_labels,
                    style=classification,
                    palette="deep",
                    legend=None
                    )
    plt.plot([x for x, _ in kmeans.centroids],
            [y for _, y in kmeans.centroids],
            '+',
            markersize=10,
            )
    plt.show()

**Task**

Answer the questions:
* Why do we get a different clustering on every new run?
* Evaluate your implementation. What causes the poor results? (That is expected — they are supposed to be poor.)
* For which initial centroid positions can the algorithm perform badly?

*Write your thoughts here*

### Fixing the mistakes

**Task**

Change the centroid initialization lines from
```python
min_, max_ = ### YOUR CODE HERE ###
self.centroids = [uniform(min_, max_) for _ in range(self.n_clusters)]
```

to

```python
self.centroids = [random.choice(X_train)]
for _ in range(self.n_clusters-1):
    dists = np.sum([euclidean(centroid, X_train) for centroid in self.centroids], axis=0)
    dists /= np.sum(dists)
    new_centroid_idx, = np.random.choice(range(len(X_train)), size=1, p=dists)
    self.centroids += [X_train[new_centroid_idx]]
```


* Test below how your code works with this fix on the same data
* Explain what this piece of code does and why it improves the result

In [ ]:
# Тестируем здесь

### YOUR CODE HERE ###

*Explain here*

## Customer clustering and interpretability of the results (2 points)

In this task we will use the ready-made [K-Means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) implementation from sklearn. **DO NOT USE YOUR OWN**

In this part we use [The Airlines Customer Satisfaction](https://www.kaggle.com/datasets/sjleshrac/airlines-customer-satisfaction) dataset, which contains information about airline customer satisfaction. Each record is a customer, and the variables include the customer's demographics, the type of travel (business, etc.) and satisfaction ratings for various aspects of the flight.

Let's look at the data

In [2]:
import pandas as pd

airline_data = pd.read_csv('airline.csv')
airline_data.head(5)

,satisfaction,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Female,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Male,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Female,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Female,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Female,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,...,4,2,2,0,2,4,2,5,0,0.0


In [4]:
airline_data.isna().sum()

satisfaction                           0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Seat comfort                           0
Departure/Arrival time convenient      0
Food and drink                         0
Gate location                          0
Inflight wifi service                  0
Inflight entertainment                 0
Online support                         0
Ease of Online booking                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Online boarding                        0
Departure Delay in Minutes             0
Arrival Delay in Minutes             393
dtype: int64

### Preprocessing

Let's remove some columns that we do not want to influence our clusters — I will drop all categorical columns from the candidate solution:

* Satisfaction;
* Gender;
* Customer Type;
* Class;
* Type of Travel;

With categorical variables, we do not want too many dummy variables influencing the clusters.
As more and more binary (dummy) variables are added to a k-means solution, they start to carry a lot of weight in the final clustering distances, even after standardization. So it is important to be careful when adding this kind of data to any k-means solution.

**Task** Remove these variables from the dataset with `.drop`

In [ ]:
airline_data_filter = ### YOUR CODE HERE ###

There are also missing values in "Arrival Delay in Minutes"; let's fill them in.

In [ ]:
airline_data_filter['Arrival Delay in Minutes'] = np.where(
    airline_data_filter['Arrival Delay in Minutes'].isna(),
    airline_data_filter['Departure Delay in Minutes'],
    airline_data_filter['Arrival Delay in Minutes']
)

**Task** Later on it will be hard to analyse the clustering results because our feature vectors are very high-dimensional. Moreover, high dimensionality often hurts K-Means.

To get rid of some features, we can use a familiar trick: remove or combine highly correlated features.

To do this, plot a heatmap of the feature correlations.

*Hint*: `dp.corr()` and `sns.heatmap` will help

In [ ]:
# Строим тепловую карту корреляции признаков

### YOUR CODE HERE ###

**Task** Now take the mean of the groups of features that are highly correlated with each other.

Create the new features:
* Online and Wi-Fi Satisfaction — the mean of 'Online boarding', 'Inflight wifi service', 'Online support', 'Ease of Online booking'
* Comfort & Food — the mean of 'Seat comfort', 'Food and drink'

(do not forget to drop the features used in the new variables so they are not duplicated)

In [ ]:
### YOUR CODE HERE ###

The next preprocessing step is standardization: we need the different features to have the same scale, i.e. all variables should take values from 0 to 1.

This can be done with [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) from sklearn.

**Task**  Explain in your own words why standardization matters for K-Means.

*Explain here*

**Task**  Use StandardScaler

In [ ]:
scaled_airline = ### YOUR CODE HERE ###

Hooray! You did it! Now we can apply the algorithm to our data!

### K-means

Let's train a KMeans model on our data and see how it splits the data into clusters.


But first we need to decide how many clusters to use. For this we will use the elbow method. \
The method is to plot how $J^{class}$ changes with the number of clusters and see at which number of clusters the rate of decrease bends.

**Task** Try numbers of clusters from 2 to 12, train K-Means for each and plot how the metric changes.

*Hint*: anyone who has read the sklearn K-Means documentation carefully already knows that this metric is stored in the `inertia_` attribute after training.

In [ ]:
### YOUR CODE HERE ###

**Task** Explain in your own words why we need the elbow method and which number of clusters you chose (and why). Why can't we just pick some small or some large number of clusters?

*Your explanation here*

**Task** Now that we have chosen the number of clusters, let's train the model and get the predictions.

In [ ]:
kmeans = ### YOUR CODE HERE ###
airline_data_filter['cluster_kmeans'] = kmeans.predict(scaled_airline)

### Analysing the clustering results

Now that we have clustered the data, it is time to see which clusters the algorithm produced. This information helps us understand the main groups of our customers and the problems they care about.

To do this, compare the mean value of each feature in every cluster with the mean value across all customers.

In [ ]:
airline_data_filter.groupby(['cluster_kmeans']).mean()

In [ ]:
airline_data_filter.mean()

**Task** Describe in detail the key characteristics of each cluster

*Your description here*

**Task** Suggest ideas for raising the ratings of each group of customers.

Personalized offers can be a good idea: remember that we want to save the company money, so do not hand out discounts to those who do not need them.

*Your suggestions here*

# Free swim

**This is a creative assignment**


**The student with the best submission automatically gets top marks for the next test**

Danya is a student and a keen traveller who dreams of visiting as many cities as possible this summer. But he has one quirk: he is extremely picky about the climate. If the weather during a trip turns out too cold or, on the contrary, too hot, he will be upset. Danya cannot describe exactly what weather he likes, but he can name one city where the summer temperature is perfect for him.

Your goal is to help Danya choose cities with a climate he finds comfortable, using k-means clustering. We will try to split cities into clusters with similar climate conditions to build a list of places where Danya would enjoy spending the summer.

*Requirements*

1. Climate homogeneity: within each cluster, cities should be as close as possible in their weather conditions, so that Danya does not run into any climate surprises.
2. Cluster size: each cluster should contain enough cities for Danya to choose from and plan a full itinerary.

Use the [World Weather Repository](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository) dataset. It contains daily historical weather data for cities over the last 5 months (about 40 different features). You need to cluster these cities by weather.

(OK, not entirely a free swim)
Plan:
1. Data preprocessing
    1. Filter records: keep only records from the summer period (June, July, August).
    2. Feature selection: you do not have to use every feature. Think about which features are useful; for example, `temperature_fahrenheit` is useless if you already use `temperature_celsius`.
    3. Aggregate by city: combine the historical records for each city. For numerical features such as temperature you can take, for example, the minimum, maximum and mean.
    4. Feature standardization: transform the data so that all features are on roughly the same scale. This makes the clustering more accurate.
2. K-Means
    1. Choose the number of clusters: use the elbow method to find the optimal value of k (the number of clusters).
    2. Train the model: run k-means with the chosen k.
    3. Analyse the clusters: examine the results of the algorithm and change the number of clusters if necessary.
3. Visualize the results: show the climate-based split of cities on a map. You are free to do whatever you like here — experiment with different kinds of plots.

Useful resources for inspiration:
* [Folium basics](https://medium.com/@enduranceprog/creating-a-python-web-map-5d5a5c081078)
* [Working with geodata in Python and Jupyter, in detail (in Russian)](https://proglib.io/p/rabota-s-geodannymi-v-python-i-jupyter-2021-03-22)
* [Ten simple folium examples](https://snyk.io/advisor/python/folium/example)